## Silver Layer: Data Cleansing & Feature Engineering

This notebook cleans, validates, and enriches raw records from the Bronze layer to build a trusted, analytics-ready Silver Delta table.

### Key Objectives & Operations:

* **Schema Standardization:** Renaming clinical abbreviations into standardized descriptors (e.g., `ap_hi` → `systolic_bp`, `ap_lo` → `diastolic_bp`).
* **Physiological Validation:** Filtering out medically impossible readings (`diastolic_bp <= systolic_bp`, negative pressures, and extreme BP/BMI outliers).
* **Clinical Feature Engineering:** Deriving physiological metrics including `age_years`, `BMI`, Pulse Pressure, Mean Arterial Pressure (`map`), and blood pressure staging categories.
* **Data Quality Checks:** Verifying zero duplicate records and handling missing values across all vital signs.
* **Target Destination:** Overwriting/merging into the curated Silver Delta table (`cardio_sil`) for modeling and dashboard queries.

In [0]:
from pyspark.sql.functions import col
df=spark.table('cardio_br')

In [0]:
df=df.withColumnRenamed('ap_hi','systolic_bp')\
.withColumnRenamed('ap_lo','diastolic_bp')

Removing invalid bp values

In [0]:
df=df.filter(col('diastolic_bp')<=col('systolic_bp'))

In [0]:
df=df.filter((col('systolic_bp')>=60) & (col('systolic_bp')<=250)&
(col('diastolic_bp')>=40) & (col('diastolic_bp')<=150))


In [0]:
df=df.dropDuplicates().na.drop()

In [0]:
df=df.withColumn('age_years',col('age')/365)

In [0]:
df.show(10)

+---+-----+------+------+------+-----------+------------+-----------+----+-----+----+------+------+------------------+
| id|  age|gender|height|weight|systolic_bp|diastolic_bp|cholesterol|gluc|smoke|alco|active|cardio|         age_years|
+---+-----+------+------+------+-----------+------------+-----------+----+-----+----+------+------+------------------+
|  0|18393|     2|   168|  62.0|        110|          80|          1|   1|    0|   0|     1|     0|50.391780821917806|
|  1|20228|     1|   156|  85.0|        140|          90|          3|   1|    0|   0|     1|     1| 55.41917808219178|
|  2|18857|     1|   165|  64.0|        130|          70|          3|   1|    0|   0|     0|     1| 51.66301369863014|
|  3|17623|     2|   169|  82.0|        150|         100|          1|   1|    0|   0|     1|     1| 48.28219178082192|
|  4|17474|     1|   156|  56.0|        100|          60|          1|   1|    0|   0|     0|     0| 47.87397260273973|
|  8|21914|     1|   151|  67.0|        120|    

In [0]:
df=df.drop('age')

In [0]:
from pyspark.sql.functions import col, round
df=df.withColumn("age_years", round(col("age_years")))

In [0]:
from pyspark.sql.functions import col
df=df.withColumn("height_m",col("height")/100)
df=df.withColumn("BMI",col("weight")/(col("height_m") ** 2))
df=df.drop('height_m')

In [0]:
df.show(10)

+---+------+------+------+-----------+------------+-----------+----+-----+----+------+------+---------+------------------+
| id|gender|height|weight|systolic_bp|diastolic_bp|cholesterol|gluc|smoke|alco|active|cardio|age_years|               BMI|
+---+------+------+------+-----------+------------+-----------+----+-----+----+------+------+---------+------------------+
|  0|     2|   168|  62.0|        110|          80|          1|   1|    0|   0|     1|     0|     50.0|  21.9671201814059|
|  1|     1|   156|  85.0|        140|          90|          3|   1|    0|   0|     1|     1|     55.0|34.927679158448385|
|  2|     1|   165|  64.0|        130|          70|          3|   1|    0|   0|     0|     1|     52.0|23.507805325987146|
|  3|     2|   169|  82.0|        150|         100|          1|   1|    0|   0|     1|     1|     48.0| 28.71047932495361|
|  4|     1|   156|  56.0|        100|          60|          1|   1|    0|   0|     0|     0|     48.0|23.011176857330703|
|  8|     1|   1

In [0]:
from pyspark.sql.functions import when
df = df.withColumn(
    "BP_Category",
    when((col("systolic_bp") < 120) & (col("diastolic_bp") < 80), "Normal")
    .when((col("systolic_bp").between(120,129)) & (col("diastolic_bp") < 80), "Elevated")
    .when((col("systolic_bp") >= 130) | (col("diastolic_bp") >= 80), "Hypertension")
    .otherwise("Unknown")
)

In [0]:
df.write.mode('overwrite').saveAsTable('cardio_sil')